# Diagnóstico posterior da matheurística determinística

Este notebook **somente lê resultados**. Não instancia nem resolve modelos de otimização e não é executado pelo pipeline.
Escolha uma execução por `run_id`; índices de produto, veículo e período começam em **zero**, e o depósito é o nó **0**.
`u` e `eta` são gravadas sempre que houver uma solução na etapa. Variáveis físicas intermediárias e auxiliares de metas exigem `solver.matheuristic.debug.enabled=true`.
Zeros omitidos de uma família marcada `sparse=true` são zeros. Uma família ausente **não** é uma matriz de zeros: significa que ela não foi salva.

In [1]:
from pathlib import Path
from types import SimpleNamespace
import json
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

REPO = Path.cwd() if (Path.cwd()/"src").exists() else Path.cwd().parent
sys.path.insert(0,str(REPO))
from src.solvers._solver_common import ProblemData, solution_components, evaluate_configured_objective
from src.solvers._routing_common import routes_from_z, ordered_arcs, validate_variables
from src.helpers.ReadPrpFile import ReadPrpFile

OUTPUT_ROOT = REPO / "out"  # Ajuste para o diretório dos resultados.
RUN_ID = '084ddfd36e-matheuristic-08a0192a2a'  # Copie da tabela abaixo depois de identificar a execução desejada.
INSTANCE_FILE = None  # Opcional: novo caminho para o .dat original.
metadata_files = [
    path for path in OUTPUT_ROOT.rglob("*.metadata.json")
    if path.parent.name == "matheuristic"
]
run_rows = []
for meta_path in metadata_files:
    metadata = json.loads(meta_path.read_text())
    run_id = metadata.get("run_id", meta_path.name.removesuffix(".metadata.json"))
    stage_path = meta_path.parent / f"{run_id}.stages.parquet"
    stages = pd.read_parquet(stage_path) if stage_path.exists() else pd.DataFrame()
    selected_stage = metadata.get("selected_stage")
    selected = (
        stages.loc[stages.stage.eq(selected_stage)]
        if not stages.empty and selected_stage else pd.DataFrame()
    )
    selected_row = selected.iloc[0] if not selected.empty else {}
    run_rows.append({
        "run_id": run_id,
        "arquivos_em": pd.Timestamp.fromtimestamp(meta_path.stat().st_mtime).strftime("%Y-%m-%d %H:%M:%S"),
        "instância": metadata.get("instance_file"),
        "etapa_selecionada": selected_stage,
        "status": selected_row.get("status", "sem solução selecionada"),
        "objetivo_configurado": selected_row.get("configured_objective"),
        "use_as": metadata.get("config", {}).get("solver", {}).get("matheuristic", {}).get("use_as"),
        "peso": metadata.get("weight"),
        "alpha": metadata.get("alpha"),
        "etapas": ", ".join(stages.stage.astype(str)) if not stages.empty else None,
        "commit": metadata.get("commit_hash"),
        "path": str(meta_path),
    })
inventory = pd.DataFrame(run_rows)
if not inventory.empty:
    inventory = inventory.sort_values("arquivos_em", ascending=False).reset_index(drop=True)
pd.set_option("display.max_colwidth", None)
display(inventory.drop(columns=["path"]) if "path" in inventory else inventory)

,run_id,arquivos_em,instância,etapa_selecionada,status,objetivo_configurado,use_as,peso,alpha,etapas,commit
0,084ddfd36e-matheuristic-08a0192a2a,2026-10-04 09:33:11,./data/DATA_PRP_20C/PRP13_C20_P8_V5_T12_S4.dat,restricted,time limit exceeded,28.733196,final,"[0.2, 0.2, 0.2, 0.2, 0.2]",0.99,"tsp, restricted, route_improvement",71a1bb
1,f0ee4342f6-matheuristic-beec215390,2026-10-04 09:17:07,./data/DATA_PRP_5C/PRP1_C5_P2_V1_T2_S1.dat,restricted,"integer optimal, tolerance",0.422228,final,"[0.2, 0.2, 0.2, 0.2, 0.2]",0.99,"tsp, restricted, route_improvement",0a58f8


## Carregamento e comparação das etapas

A tabela distingue `objective` (objetivo próprio da etapa) e `configured_objective` (critério para selecionar a solução).
O objetivo da melhoria é transporte; seu gap e bound pertencem à etapa `route_improvement`.
O gap de `restricted` descreve apenas a região limitada pela ordem. Nenhum deles certifica o ótimo do MPPRP irrestrito.
Em um TSP recuperado do cache, `cache_hit=true` e os tempos da execução atual são zero; bound/gap descrevem a resolução que preencheu o cache.

In [2]:
assert RUN_ID, "Copie o run_id desejado da tabela da primeira célula."
paths = inventory.loc[inventory.run_id.eq(RUN_ID),"path"].tolist()
assert len(paths) == 1, "run_id não encontrado ou duplicado."
meta_path = Path(paths[0])
meta = json.loads(meta_path.read_text())
variables_df = pd.read_parquet(meta_path.parent/f"{RUN_ID}.variables.parquet")
summary = pd.read_parquet(meta_path.parent/f"{RUN_ID}.stages.parquet")
catalog = meta["available_variables"]

silent = SimpleNamespace(debug=lambda *args:None)
data = meta.get("parameters")
if data is None:
    # O .dat original permite auditar a solução final mesmo sem debug intermediário.
    instance_path = Path(INSTANCE_FILE or meta["instance_file"])
    if not instance_path.is_absolute():
        instance_path = REPO / instance_path
    assert instance_path.exists(), "Ajuste INSTANCE_FILE para localizar o .dat original."
    data = ReadPrpFile(str(instance_path),silent).getDataSet()
    data.update(weight=meta["weight"],alpha=meta["alpha"],targets=meta["targets"],
                strengthened_bounds=meta["strengthened_bounds"])
problem = ProblemData.from_map(data)
config = meta["config"]
pi = meta["order"]
display(summary)
print("Etapa selecionada:",meta["selected_stage"],"Ordem:",pi)
fig = px.bar(summary.melt(id_vars="stage",value_vars=[c for c in
              ["build_seconds","solve_seconds","elapsed_seconds"] if c in summary],
              var_name="medida",value_name="segundos"),x="stage",y="segundos",color="medida",barmode="group")
fig.show()  # elapsed inclui solve/build; as barras não são parcelas somáveis.

,run_id,hash_file,stage,status,bound,gap,cache_hit,solve_seconds,build_seconds,variables,...,gap_scope,selected,extract_seconds,has_incumbent,has_solution,validation_errors,features,rounded_capacity,configured_objective,costs
0,084ddfd36e-matheuristic-08a0192a2a,1e26e4f613867e26b8ed2b18e8b22313bf59599d,tsp,integer optimal solution,2947.0,0.0,False,0.011703,0.029030,420,...,tsp,False,NaN,None,None,None,None,None,NaN,None
1,084ddfd36e-matheuristic-08a0192a2a,1e26e4f613867e26b8ed2b18e8b22313bf59599d,restricted,time limit exceeded,0.0,1.0,None,60.120526,0.877411,137329,...,restricted,True,0.021651,True,True,[],"{'coelho_inequalities': False, 'hc1': False, 'positive_only_deviations': True, 'rounded_capacity_callback': False, 'routing_profile': 'ordered', 'strengthened_bounds': False, 'visit_delivery_bounds': True}","{'callback_calls': 0.0, 'callback_errors': 0.0, 'callback_registered': False, 'candidate_evaluations': 0.0, 'cuts_added': 0.0, 'cuts_found': 0.0, 'cuts_rejected': 0.0, 'duplicate_cuts': 0.0, 'max_violation': 0.0, 'separator_average_seconds': 0.0, 'separator_calls': 0.0, 'separator_seconds': 0.0}",28.733196,"[[13655091.796875, 13527745.216369629, 13367813.528865576, 13437778.96490062, 13597601.986207047, 13597243.398962859, 13484482.621032981, 13610493.903576152, 15438454.98738971, 16863170.145677563, 17546616.06351549, 17716786.67968538], [4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0], [177822.91796875, 266018.3701324463, 353836.50542110205, 437416.29507010826, 521663.31493217876, 609933.7489218073, 697873.5751321373, 787663.5141678987, 889059.0640417958, 1009143.7654985714, 1137085.9261337263, 1262198.79293058], [0.0, 40068.0, 60102.0, 80136.0, 80136.0, 80136.0, 80136.0, 80136.0, 80136.0, 80136.0, 80136.0, 80136.0], [0.0, 4729.0, 5662.0, 6787.0, 7453.0, 7434.0, 6792.0, 7284.0, 7172.0, 6693.0, 6801.0, 7185.0]]"
2,084ddfd36e-matheuristic-08a0192a2a,1e26e4f613867e26b8ed2b18e8b22313bf59599d,route_improvement,time limit exceeded,0.0,1.0,None,30.181984,1.377037,221029,...,route_improvement,False,0.037548,True,True,[],"{'coelho_inequalities': False, 'hc1': False, 'positive_only_deviations': True, 'rounded_capacity_callback': False, 'routing_profile': 'improvement', 'strengthened_bounds': False, 'visit_delivery_bounds': True}","{'callback_calls': 0.0, 'callback_errors': 0.0, 'callback_registered': False, 'candidate_evaluations': 0.0, 'cuts_added': 0.0, 'cuts_found': 0.0, 'cuts_rejected': 0.0, 'duplicate_cuts': 0.0, 'max_violation': 0.0, 'separator_average_seconds': 0.0, 'separator_calls': 0.0, 'separator_seconds': 0.0}",28.733196,"[[13655091.796875, 13527745.216369629, 13367813.528865576, 13437778.96490062, 13597601.986207047, 13597243.398962859, 13484482.621032981, 13610493.903576152, 15438454.98738971, 16863170.145677563, 17546616.06351549, 17716786.67968538], [4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0, 4400.0], [177822.91796875, 266018.3701324463, 353836.50542110205, 437416.29507010826, 521663.31493217876, 609933.7489218073, 697873.5751321373, 787663.5141678987, 889059.0640417958, 1009143.7654985714, 1137085.9261337263, 1262198.79293058], [0.0, 40068.0, 60102.0, 80136.0, 80136.0, 80136.0, 80136.0, 80136.0, 80136.0, 80136.0, 80136.0, 80136.0], [0.0, 4729.0, 5662.0, 6787.0, 7453.0, 7434.0, 6792.0, 7284.0, 7172.0, 6693.0, 6801.0, 7185.0]]"


Etapa selecionada: restricted Ordem: [0, 14, 4, 20, 7, 10, 13, 17, 5, 2, 6, 1, 18, 16, 9, 19, 11, 3, 12, 8, 15, 0]


In [3]:
def restore(stage,var):
    spec = catalog.get(stage,{}).get(var)
    if spec is None:
        raise KeyError(f"{stage}/{var} não foi salva; habilite debug para estados intermediários.")
    array = np.zeros(spec["shape"],dtype=np.float64)
    rows = variables_df.loc[variables_df.stage.eq(stage)&variables_df["var"].eq(var)]
    if not spec["indices"]:
        return np.array(rows.value.iloc[0] if len(rows) else 0.)
    for row in rows.itertuples(index=False):
        key = tuple(int(getattr(row,index)) for index in spec["indices"])
        array[key] = row.value
    return array

physical = ("X","Y","I","Q","R","Z")
states = {stage:{var:restore(stage,var) for var in (*physical,"eta") if var in families}
          for stage,families in catalog.items() if all(var in families for var in physical)}
# Recupera também a saída final no contrato legado, identificada pelo mesmo hash.
final_path = meta_path.parent.parent/"parquets"/f"{meta['hash_file'][:6]}_fobs.parquet"
stamp = meta.get('final_output')
final_available = (final_path.exists() and stamp is not None and
                   final_path.stat().st_mtime_ns == stamp['mtime_ns'] and final_path.stat().st_size == stamp['size'])
if final_available:
    final_df = pd.read_parquet(final_path)
    assert final_df.empty or set(final_df.hash_file) == {meta["hash_file"]}, "Arquivo final não corresponde à execução."
    shapes = {"X":(problem.p,problem.t),"Y":(problem.p,problem.t),
              "I":(problem.p,problem.i,problem.t),"Q":(problem.p,problem.v,problem.i,problem.t),
              "R":(problem.p,problem.v,problem.i,problem.k,problem.t),"Z":(problem.v,problem.i,problem.k,problem.t)}
    indices = {"X":("p","t"),"Y":("p","t"),"I":("p","i","t"),"Q":("p","v","i","t"),
               "R":("p","v","i","k","t"),"Z":("v","i","k","t")}
    if all(var in set(final_df.get('var',pd.Series(dtype=str))) for var in physical):
        state = {var:np.zeros(shape) for var,shape in shapes.items()}
        for var,keys in indices.items():
            for row in final_df.loc[final_df['var'].eq(var)].itertuples(index=False):
                state[var][tuple(int(getattr(row,key)) for key in keys)] = row.value
        states['final'] = state
if final_path.exists() and not final_available:
    print("Saída final ausente desta execução ou sobrescrita; use os snapshots de debug.")
print("Estados físicos disponíveis:",list(states))

Estados físicos disponíveis: ['restricted', 'route_improvement', 'final']


## TSP, visitas, rotas e cargas

A ordem do TSP é comum aos produtos, veículos e períodos. A tabela de rotas mostra subsequências na etapa restrita e sequências livres na melhoria.
Escolha `STAGE` e `PERIOD` para inspeção. Os gráficos usam as coordenadas da instância; sem coordenadas, empregam uma disposição circular apenas para visualização, sem alterar custos.

In [4]:
def plot_routes(routes, title):
    coord = data.get('coordXY')
    if coord:
        x, y = np.asarray(coord['x']), np.asarray(coord['y'])
    else:
        angle = np.linspace(0, 2 * np.pi, problem.i, endpoint=False)
        x, y = np.cos(angle), np.sin(angle)

    fig = go.Figure()
    palette = px.colors.qualitative.Plotly
    for vehicle, route in enumerate(routes):
        if not route:
            continue
        nodes = [0, *route, 0]
        color = palette[vehicle % len(palette)]
        fig.add_trace(
            go.Scatter(
                x=x[nodes],
                y=y[nodes],
                mode='lines+markers',
                line={'color': color, 'width': 2},
                marker={'size': 8},
                name=f'veículo {vehicle}',
                text=[f'parada {step}: nó {node}' for step, node in enumerate(nodes)],
                hovertemplate='%{text}<extra>%{fullData.name}</extra>',
            )
        )
        for step, (origin, destination) in enumerate(zip(nodes, nodes[1:]), start=1):
            dx, dy = x[destination] - x[origin], y[destination] - y[origin]
            # A seta aponta no sentido percorrido; o número identifica a perna da rota.
            fig.add_annotation(
                x=x[origin] + 0.68 * dx,
                y=y[origin] + 0.68 * dy,
                ax=x[origin] + 0.43 * dx,
                ay=y[origin] + 0.43 * dy,
                xref='x', yref='y', axref='x', ayref='y',
                text='', showarrow=True, arrowhead=3, arrowsize=1.2,
                arrowwidth=2, arrowcolor=color,
            )
            fig.add_annotation(
                x=x[origin] + 0.38 * dx,
                y=y[origin] + 0.38 * dy,
                text=str(step), showarrow=False,
                font={'color': color, 'size': 11},
                bgcolor='white', bordercolor=color, borderwidth=1,
            )

    fig.add_trace(
        go.Scatter(
            x=x, y=y, mode='markers+text', text=list(range(problem.i)),
            textposition='top center', name='nós',
            marker={'color': 'black', 'size': 10},
        )
    )
    fig.update_layout(
        title=title,
        yaxis_scaleanchor='x',
        annotations=fig.layout.annotations,
        legend_title_text='Ordem por veículo',
    )
    return fig

u = restore('tsp','u')
assert np.allclose(u,np.rint(u))
assert all(u[a,b] == 1 for a,b in zip(pi,pi[1:]) if a != b)
plot_routes([pi[1:-1]],'TSP: ordem comum').show()
route_records = []
for stage,state in states.items():
    try:
        for t,period in enumerate(routes_from_z(state['Z'])):
            for v,route in enumerate(period):
                route_records.append({'stage':stage,'t':t,'v':v,'route':[0,*route,0] if route else [],
                                      'sequence':' → '.join(map(str,[0,*route,0])) if route else '',
                                      'delivered':float(state['Q'][:,v,:,t].sum())})
    except ValueError as error:
        print(stage,'Falha de conectividade:',error)
display(pd.DataFrame(route_records))

STAGE = next(iter(states),None)  # Ajuste para restricted, route_improvement ou final.
STAGE = 'restricted'  # Ajuste para restricted, route_improvement ou final.
PERIOD = 0
if STAGE:
    state = states[STAGE]
    routes = routes_from_z(state['Z'])[PERIOD]
    plot_routes(routes,f'{STAGE}, período {PERIOD}').show()
    rows = []
    for v,route in enumerate(routes):
        for i,k in zip([0,*route],[*route,0]) if route else []:
            rows.append({'v':v,'i':i,'k':k,'Z':state['Z'][v,i,k,PERIOD],
                         'carga_por_produto':state['R'][:,v,i,k,PERIOD].tolist(),
                         'carga_total':state['R'][:,v,i,k,PERIOD].sum()})
    display(pd.DataFrame(rows))

,stage,t,v,route,sequence,delivered
0,restricted,0,0,[],,0.0
1,restricted,0,1,[],,0.0
2,restricted,0,2,[],,0.0
3,restricted,0,3,[],,0.0
4,restricted,0,4,[],,0.0
...,...,...,...,...,...,...
175,final,11,0,"[0, 4, 7, 17, 16, 8, 0]",0 → 4 → 7 → 17 → 16 → 8 → 0,2684.0
176,final,11,1,"[0, 20, 5, 1, 18, 15, 0]",0 → 20 → 5 → 1 → 18 → 15 → 0,2552.0
177,final,11,2,"[0, 14, 10, 13, 19, 11, 3, 0]",0 → 14 → 10 → 13 → 19 → 11 → 3 → 0,2693.0
178,final,11,3,"[0, 2, 6, 9, 12, 0]",0 → 2 → 6 → 9 → 12 → 0,1526.0


""


In [5]:
aaa = pd.DataFrame(route_records)
aaa.query('stage == "restricted"')

,stage,t,v,route,sequence,delivered
0,restricted,0,0,[],,0.0
1,restricted,0,1,[],,0.0
2,restricted,0,2,[],,0.0
3,restricted,0,3,[],,0.0
4,restricted,0,4,[],,0.0
5,restricted,1,0,"[0, 14, 4, 7, 10, 17, 5, 6, 16, 9, 11, 12, 15, 0]",0 → 14 → 4 → 7 → 10 → 17 → 5 → 6 → 16 → 9 → 11 → 12 → 15 → 0,2674.0
6,restricted,1,1,"[0, 20, 13, 2, 1, 18, 19, 3, 8, 0]",0 → 20 → 13 → 2 → 1 → 18 → 19 → 3 → 8 → 0,770.0
7,restricted,1,2,[],,0.0
8,restricted,1,3,[],,0.0
9,restricted,1,4,[],,0.0


## Auditoria numérica

A auditoria verifica domínios, balanços de estoque/fluxo, ligação carga/arco, capacidade de produção/estoque e entrega sem visita.
A conectividade é verificada inclusive para visitas com entrega zero. Não é usada tolerância para classificar uma entrega positiva da etapa 3; a auditoria de resíduos usa tolerância numérica.
A tabela explicita os resíduos máximos, inclusive `eta/Z`, e os custos são recalculados com a mesma avaliação usada no pipeline.

In [6]:
audits,cost_records = [],[]
for stage,state in states.items():
    report = validate_variables(problem,state,ordered_arcs(pi) if stage == 'restricted' else None)
    previous = np.concatenate((np.asarray(problem.I_p_i_0)[:,:,None],state['I'][:,:,:-1]),axis=2)
    plant_res = previous[:,0]+state['X']-state['Q'][:,:,1:].sum(axis=(1,2))-state['I'][:,0]
    customer_res = previous[:,1:]+state['Q'][:,:,1:].sum(axis=1)-problem.d_p_i_t-state['I'][:,1:]
    flow_res = state['R'].sum(axis=2)[:,:,1:]-state['R'].sum(axis=3)[:,:,1:]-state['Q'][:,:,1:]
    eta_out = eta_in = np.nan
    if 'eta' in state:
        eta_out = np.max(np.abs(state['Z'].sum(axis=2)-state['eta']))
        eta_in = np.max(np.abs(state['Z'].sum(axis=1)-state['eta']))
    audits.append({'stage':stage,**report,'plant_residual':np.max(np.abs(plant_res)),
                   'customer_residual':np.max(np.abs(customer_res)), 'flow_residual':np.max(np.abs(flow_res)),
                   'eta_out_residual':eta_out,'eta_in_residual':eta_in,
                   'capacity_excess':np.maximum(state['R'].sum(axis=0)-problem.C*state['Z'],0).max()})
    costs = solution_components(problem,state)
    cost_records.append({'stage':stage,'configured_objective':evaluate_configured_objective(problem,state,config),
                         **{f'F{j+1}':float(costs[j].sum()) for j in range(5)}})
display(pd.DataFrame(audits))
display(pd.DataFrame(cost_records))
if 'restricted' in states and 'route_improvement' in states:
    before,after = states['restricted'],states['route_improvement']
    display(pd.DataFrame([{'quantity':name,'max_change':np.max(np.abs(before[name]-after[name]))} for name in ('X','Y','I')]
        +[{'quantity':'Q agregado','max_change':np.max(np.abs(before['Q'].sum(axis=1)-after['Q'].sum(axis=1)))}]))
    for t in range(problem.t):
        active = after['Q'].sum(axis=(0,1))[:,t] > 0
        visited = after['Z'].sum(axis=(0,2))[:,t] > .5
        assert np.array_equal(active[1:],visited[1:]), f'Visitas indevidas na melhoria, período {t}'

,stage,feasible,violations,plant_residual,customer_residual,flow_residual,eta_out_residual,eta_in_residual,capacity_excess
0,restricted,True,[],0.0,0.0,0.0,0.0,0.0,0.0
1,route_improvement,True,[],0.0,0.0,0.0,0.0,0.0,0.0
2,final,True,[],0.0,0.0,0.0,NaN,NaN,0.0


,stage,configured_objective,F1,F2,F3,F4,F5
0,restricted,28.733196,1.758433e+08,52800.0,8.149716e+06,821394.0,73992.0
1,route_improvement,28.733196,1.758433e+08,52800.0,8.149716e+06,821394.0,73992.0
2,final,28.733196,1.758433e+08,52800.0,8.149716e+06,821394.0,73992.0


,quantity,max_change
0,X,0.0
1,Y,0.0
2,I,0.0
3,Q agregado,0.0


## Atendimento da demanda

Para cada estágio, esta seção compara estoque inicial no cliente, entregas `Q`, demanda, atendimento calculado, falta e estoque final. O balanço é `estoque inicial + entregas - demanda = estoque final`; como o modelo não tem variável de falta, uma solução factível deve apresentar falta calculada igual a zero. A tabela detalhada usa o `STAGE` e o `PERIOD` escolhidos acima.


In [7]:
demand_summary_rows = []
demand_detail_by_stage = {}
for stage, state in states.items():
    demand = np.asarray(problem.d_p_i_t, dtype=float)
    starting_stock = np.concatenate(
        (
            np.asarray(problem.I_p_i_0, dtype=float)[:, 1:, None],
            np.asarray(state["I"], dtype=float)[:, 1:, :-1],
        ),
        axis=2,
    )
    deliveries = np.asarray(state["Q"], dtype=float)[:, :, 1:, :].sum(axis=1)
    ending_stock = np.asarray(state["I"], dtype=float)[:, 1:, :]
    available = starting_stock + deliveries
    attended = np.minimum(demand, np.maximum(available, 0.0))
    shortage = np.maximum(demand - available, 0.0)
    balance_residual = starting_stock + deliveries - demand - ending_stock

    for t in range(problem.t):
        period_demand = float(demand[:, :, t].sum())
        period_shortage = float(shortage[:, :, t].sum())
        demand_summary_rows.append({
            "stage": stage,
            "periodo": t,
            "estoque_inicial_clientes": float(starting_stock[:, :, t].sum()),
            "entregas_Q": float(deliveries[:, :, t].sum()),
            "demanda": period_demand,
            "demanda_atendida": float(attended[:, :, t].sum()),
            "falta_calculada": period_shortage,
            "estoque_final_clientes": float(ending_stock[:, :, t].sum()),
            "atendimento_pct": (
                100.0 * (period_demand - period_shortage) / period_demand
                if period_demand > 0 else np.nan
            ),
            "residuo_max_balanco": float(np.abs(balance_residual[:, :, t]).max()),
        })

    demand_detail_by_stage[stage] = pd.DataFrame([
        {
            "produto_p": p,
            "cliente_no": i + 1,
            "periodo": t,
            "estoque_inicial": float(starting_stock[p, i, t]),
            "entrega_Q": float(deliveries[p, i, t]),
            "demanda": float(demand[p, i, t]),
            "atendida": float(attended[p, i, t]),
            "falta_calculada": float(shortage[p, i, t]),
            "estoque_final": float(ending_stock[p, i, t]),
            "residuo_balanco": float(balance_residual[p, i, t]),
        }
        for p in range(problem.p)
        for i in range(problem.i - 1)
        for t in range(problem.t)
    ])

demand_summary = pd.DataFrame(demand_summary_rows)
display(demand_summary.loc[demand_summary.stage.eq(STAGE)])
if STAGE in demand_detail_by_stage:
    detail = demand_detail_by_stage[STAGE]
    display(detail.loc[detail.periodo.eq(PERIOD)].reset_index(drop=True))
    selected_period = demand_summary.loc[
        demand_summary.stage.eq(STAGE)
    ].sort_values("periodo")
    fig = px.line(
        selected_period,
        x="periodo",
        y=["demanda", "demanda_atendida"],
        markers=True,
        title=f"Demanda e atendimento por período: {STAGE}",
    )
    fig.show()
    fig = px.bar(
        selected_period,
        x="periodo",
        y="entregas_Q",
        title=f"Entregas por período: {STAGE}",
    )
    fig.show()


,stage,periodo,estoque_inicial_clientes,entregas_Q,demanda,demanda_atendida,falta_calculada,estoque_final_clientes,atendimento_pct,residuo_max_balanco
0,restricted,0,17540.0,0.0,8792.0,8792.0,0.0,8748.0,100.0,0.0
1,restricted,1,8748.0,3444.0,8986.0,8986.0,0.0,3206.0,100.0,0.0
2,restricted,2,3206.0,6573.0,8832.0,8832.0,0.0,947.0,100.0,0.0
3,restricted,3,947.0,8334.0,9281.0,9281.0,0.0,0.0,100.0,0.0
4,restricted,4,0.0,9338.0,9338.0,9338.0,0.0,0.0,100.0,0.0
5,restricted,5,0.0,8469.0,8469.0,8469.0,0.0,0.0,100.0,0.0
6,restricted,6,0.0,8570.0,8570.0,8570.0,0.0,0.0,100.0,0.0
7,restricted,7,0.0,8505.0,8505.0,8505.0,0.0,0.0,100.0,0.0
8,restricted,8,0.0,9713.0,9713.0,9713.0,0.0,0.0,100.0,0.0
9,restricted,9,0.0,8634.0,8634.0,8634.0,0.0,0.0,100.0,0.0


,produto_p,cliente_no,periodo,estoque_inicial,entrega_Q,demanda,atendida,falta_calculada,estoque_final,residuo_balanco
0,0,1,0,153.0,0.0,55.0,55.0,0.0,98.0,0.0
1,0,2,0,55.0,0.0,55.0,55.0,0.0,0.0,0.0
2,0,3,0,119.0,0.0,27.0,27.0,0.0,92.0,0.0
3,0,4,0,217.0,0.0,82.0,82.0,0.0,135.0,0.0
4,0,5,0,195.0,0.0,18.0,18.0,0.0,177.0,0.0
...,...,...,...,...,...,...,...,...,...,...
155,7,16,0,76.0,0.0,76.0,76.0,0.0,0.0,0.0
156,7,17,0,29.0,0.0,29.0,29.0,0.0,0.0,0.0
157,7,18,0,71.0,0.0,36.0,36.0,0.0,35.0,0.0
158,7,19,0,144.0,0.0,44.0,44.0,0.0,100.0,0.0


## Produção, estoques e entregas

As quantidades são contínuas. Escolha um produto e período para inspecionar seus valores fracionários.
As tabelas de `P`, `N` e `lambda` mostram os auxiliares efetivamente utilizados pelo solver quando o debug os salvou.

In [8]:
PRODUCT = 0
if STAGE:
    state = states[STAGE]
    production = pd.DataFrame({'t':range(problem.t),'X':state['X'][PRODUCT],'Y':state['Y'][PRODUCT]})
    display(production)
    px.line(production,x='t',y='X',markers=True,title=f'Produção: {STAGE}, produto {PRODUCT}').show()
    px.imshow(state['I'][PRODUCT],labels={'x':'período','y':'nó','color':'estoque'},title=f'Estoque: produto {PRODUCT}').show()
    delivery = state['Q'][PRODUCT,:,:,PERIOD]
    display(pd.DataFrame(delivery,index=[f'veículo {v}' for v in range(problem.v)]))
    px.imshow(delivery,labels={'x':'nó','y':'veículo','color':'entrega'},title=f'Entregas: período {PERIOD}').show()
for stage,families in catalog.items():
    for var in ('P','N','lambda'):
        if var in families:
            print(stage,var)
            display(restore(stage,var))

,t,X,Y
0,0,13393.500000,1.0
1,1,13089.955078,1.0
2,2,12602.201897,1.0
3,3,12352.091458,1.0
4,4,11804.682139,1.0
5,5,12027.680594,1.0
6,6,11954.246910,1.0
7,7,11876.944155,1.0
8,8,7184.348884,1.0
9,9,4256.174442,1.0


,0,1,2,3,4,5,6,7,8,9,...,11,12,13,14,15,16,17,18,19,20
veículo 0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
veículo 1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
veículo 2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
veículo 3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
veículo 4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


restricted P


array([[8.51966180e+06, 1.18868952e+07, 1.03284535e+07, 8.07364896e+06,
        1.01991920e+07, 9.40426340e+06, 9.50024262e+06, 1.01479839e+07,
        1.26521150e+07, 1.47353901e+07, 1.38922461e+07, 1.45220367e+07],
       [3.07139601e+03, 3.04490788e+03, 3.06097911e+03, 3.01852755e+03,
        3.06717568e+03, 3.00653489e+03, 3.04138201e+03, 3.05659088e+03,
        3.07131958e+03, 3.07373418e+03, 3.05990051e+03, 3.06336105e+03],
       [8.63684180e+04, 2.07774170e+05, 3.08253605e+05, 3.94996895e+05,
        4.72697015e+05, 5.71826849e+05, 6.59104275e+05, 7.51275314e+05,
        8.65531564e+05, 9.96867765e+05, 1.12904623e+06, 1.26219879e+06],
       [0.00000000e+00, 1.20345189e+04, 1.05096759e+04, 1.52840991e+04,
        2.64439883e+04, 1.91957822e+04, 2.56956857e+04, 1.35051156e+04,
        1.00125414e+04, 3.35168226e+04, 1.67844198e+04, 2.91897461e+04],
       [0.00000000e+00, 2.86516536e+03, 2.68437904e+03, 2.64880156e+03,
        4.81556947e+03, 3.68852344e+03, 4.18300716e+03, 3.29

restricted lambda


array(28.08677503)

route_improvement P


array([[8.51966180e+06, 1.18868952e+07, 1.03284535e+07, 8.07364896e+06,
        1.01991920e+07, 9.40426340e+06, 9.50024262e+06, 1.01479839e+07,
        1.26521150e+07, 1.47353901e+07, 1.38922461e+07, 1.45220367e+07],
       [3.07139601e+03, 3.04490788e+03, 3.06097911e+03, 3.01852755e+03,
        3.06717568e+03, 3.00653489e+03, 3.04138201e+03, 3.05659088e+03,
        3.07131958e+03, 3.07373418e+03, 3.05990051e+03, 3.06336105e+03],
       [8.63684180e+04, 2.07774170e+05, 3.08253605e+05, 3.94996895e+05,
        4.72697015e+05, 5.71826849e+05, 6.59104275e+05, 7.51275314e+05,
        8.65531564e+05, 9.96867765e+05, 1.12904623e+06, 1.26219879e+06],
       [0.00000000e+00, 1.20345189e+04, 1.05096759e+04, 1.52840991e+04,
        2.64439883e+04, 1.91957822e+04, 2.56956857e+04, 1.35051156e+04,
        1.00125414e+04, 3.35168226e+04, 1.67844198e+04, 2.91897461e+04],
       [0.00000000e+00, 2.86516536e+03, 2.68437904e+03, 2.64880156e+03,
        4.81556947e+03, 3.68852344e+03, 4.18300716e+03, 3.29

route_improvement lambda


array(28.08677503)